# 🦜 Colombian Bird Song — Model Training (SageMaker + S3)

Train a ResNet50V2-based species classifier on log-Mel spectrogram tensors fetched from S3.
Runs on AWS SageMaker Notebook Instance with feature splits stored in S3.

---

## Pipeline Overview

| Step | Description |
|------|-------------|
| 1 | Setup & install dependencies |
| 1b | Pre-flight AWS validation |
| 2 | Configuration (S3 paths & hyperparameters) |
| 3 | Fetch splits from S3 & configure paths |
| 4 | Load split & inspect class distribution |
| 5 | Build `tf.data` pipelines |
| 6 | Build model (ResNet50V2 + classification head) |
| 7 | Phase 1 — train classification head |
| 8 | Phase 2 — fine-tune top layers |
| 9 | Evaluation & diagnostics |
| 10 | Save model locally & upload to S3 |
| 11 | Inference helper (S3-compatible) |

## Step 1 — Setup

Install required packages. Run once per Colab session.

In [ ]:
%pip install tensorflow scikit-learn matplotlib seaborn boto3 --quiet

import tensorflow as tf
import boto3
print(f"TensorFlow : {tf.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")
print(f"boto3      : {boto3.__version__}")

In [ ]:
import boto3
import os

# ── Check boto3 and AWS credentials ──────────────────────────────────────
print("AWS & boto3 Configuration Check")
print("─" * 50)

try:
    # Create S3 client and verify connectivity
    s3_check = boto3.client("s3")
    
    # Try to read SageMaker execution role (if available)
    sts = boto3.client("sts")
    identity = sts.get_caller_identity()
    account_id = identity["Account"]
    arn = identity["Arn"]
    print(f"✓ AWS credentials configured")
    print(f"  Account : {account_id}")
    print(f"  ARN     : {arn}")
    
    # Infer region
    region = os.environ.get("AWS_REGION", "us-east-1")
    print(f"  Region  : {region}")
    
except Exception as e:
    print(f"✗ AWS credential error: {e}")
    print(f"  Ensure SageMaker Notebook Instance has IAM role with S3 access.")
    raise

print("\n✓ Pre-flight check complete. Ready to proceed.")

## Step 1b — Pre-flight AWS Validation

Verify AWS credentials and permissions before attempting S3 operations.
IAM role must have `s3:GetObject` (for downloads) and `s3:PutObject` (for model uploads).

## Step 2 — Configuration

**Important:** Configure S3 paths in this step before running the notebook.

- `S3_INPUT_BUCKET`: S3 path where feature folds are stored (e.g., `s3://my-bucket/bird-splits/`)
- `S3_OUTPUT_BUCKET`: S3 path where trained models will be saved (e.g., `s3://my-bucket/bird-models/`)
- `FOLD_K` (0–4): which of the 5 pre-built splits to use for this training run

IAM role must have permissions: `s3:GetObject` (input) and `s3:PutObject` (output).

In [ ]:
# ── AWS S3 Paths ─────────────────────────────────────────────────────────
# Configure these paths before running. Example:
#   S3_INPUT_BUCKET  = "s3://my-bucket/bird-splits/"
#   S3_OUTPUT_BUCKET = "s3://my-bucket/bird-models/"
S3_INPUT_BUCKET  = "s3://my-bucket/bird-splits/"      # ← UPDATE with actual S3 path
S3_OUTPUT_BUCKET = "s3://my-bucket/bird-models/"      # ← UPDATE with actual S3 path

# ── Split selection ───────────────────────────────────────────────────────
# 0–4: which of the 5 pre-built splits to use for this training run
FOLD_K = 3

# ── Data pipeline ─────────────────────────────────────────────────────────
BATCH_SIZE     = 24
SHUFFLE_BUFFER = 10_000   # set to len(train_dataset) after loading if memory allows

# ── Model ─────────────────────────────────────────────────────────────────
INPUT_HEIGHT   = 128
INPUT_WIDTH    = 188
DROPOUT_RATE   = 0.3
DENSE_UNITS    = 256

# ── Phase 1: train head only (base frozen) ────────────────────────────────
EPOCHS_PHASE1  = 30
LR_PHASE1      = 1e-3

# ── Phase 2: fine-tune top layers ─────────────────────────────────────────
EPOCHS_PHASE2  = 80
LR_PHASE2      = 1e-4       # 10× lower than phase 1
UNFREEZE_FROM  = -50        # unfreeze the last N layers of the ResNet base

# ── Callbacks ─────────────────────────────────────────────────────────────
EARLY_STOP_PATIENCE    = 6
REDUCE_LR_PATIENCE     = 3
REDUCE_LR_FACTOR       = 0.3

print("Configuration loaded.")
print(f"  Fold        : {FOLD_K}")
print(f"  Batch       : {BATCH_SIZE}")
print(f"  Phase 1     : {EPOCHS_PHASE1} epochs @ lr={LR_PHASE1}")
print(f"  Phase 2     : {EPOCHS_PHASE2} epochs @ lr={LR_PHASE2}  (unfreeze last {abs(UNFREEZE_FROM)} layers)")
print(f"  S3 input    : {S3_INPUT_BUCKET}")
print(f"  S3 output   : {S3_OUTPUT_BUCKET}")

## Step 3 — Fetch Splits from S3 & Configure Paths

Configure S3 bucket paths and download feature folds and label map.
Requires SageMaker Notebook Instance IAM role with `s3:GetObject` permission.

In [ ]:
import boto3
from pathlib import Path

# ── Initialize S3 client ──────────────────────────────────────────────────
s3_client = boto3.client("s3")

# ── Parse S3 paths ───────────────────────────────────────────────────────
def parse_s3_path(s3_path):
    """Extract bucket and prefix from s3://bucket/prefix/"""
    parts = s3_path.replace("s3://", "").rstrip("/").split("/", 1)
    bucket = parts[0]
    prefix = parts[1] if len(parts) > 1 else ""
    return bucket, prefix

s3_input_bucket, s3_input_prefix = parse_s3_path(S3_INPUT_BUCKET)
s3_output_bucket, s3_output_prefix = parse_s3_path(S3_OUTPUT_BUCKET)

# ── Normalize prefixes to always end with / for proper path construction ──
# This ensures "features" becomes "features/" so paths like features/label_map.json work correctly
if s3_input_prefix and not s3_input_prefix.endswith("/"):
    s3_input_prefix = s3_input_prefix + "/"
if s3_output_prefix and not s3_output_prefix.endswith("/"):
    s3_output_prefix = s3_output_prefix + "/"

# ── Local storage paths ──────────────────────────────────────────────────
LOCAL_BASE   = Path("/tmp")
LOCAL_SPLITS = LOCAL_BASE / "bird_splits"
LOCAL_SPLITS.mkdir(parents=True, exist_ok=True)

SPLITS_DIR = LOCAL_SPLITS
CHECKPOINT_PATH  = SPLITS_DIR / f"fold{FOLD_K}_best.weights.h5"
FINAL_MODEL_PATH = SPLITS_DIR / f"resnet50v2_fold{FOLD_K}.keras"

print(f"Local splits dir : {SPLITS_DIR}")
print(f"S3 input bucket  : {s3_input_bucket}")
print(f"S3 input prefix  : {s3_input_prefix or '(root)'}")

# ── Helper: Download file from S3 ────────────────────────────────────────
def s3_download_file(bucket, key, local_path):
    """Download a single file from S3 to local storage."""
    print(f"  Downloading s3://{bucket}/{key} → {local_path}")
    try:
        s3_client.download_file(bucket, key, str(local_path))
        print(f"    ✓ Success ({local_path.stat().st_size / 1_048_576:.1f} MB)")
    except Exception as e:
        print(f"    ✗ Failed: {e}")
        raise

# ── Download label_map.json ─────────────────────────────────────────────
label_map_key = f"{s3_input_prefix}label_map.json".lstrip("/")
label_map_path = SPLITS_DIR / "label_map.json"
print(f"\nDownloading label map...")
s3_download_file(s3_input_bucket, label_map_key, label_map_path)

# ── Download train and val splits ───────────────────────────────────────
print(f"\nDownloading splits for fold {FOLD_K}...")
train_key = f"{s3_input_prefix}split_{FOLD_K}_train.npz".lstrip("/")
val_key   = f"{s3_input_prefix}split_{FOLD_K}_val.npz".lstrip("/")

train_path = SPLITS_DIR / f"split_{FOLD_K}_train.npz"
val_path   = SPLITS_DIR / f"split_{FOLD_K}_val.npz"

s3_download_file(s3_input_bucket, train_key, train_path)
s3_download_file(s3_input_bucket, val_key,   val_path)

# ── Verify downloads ────────────────────────────────────────────────────
print(f"\nVerifying downloaded files...")
for fpath in [label_map_path, train_path, val_path]:
    if not fpath.exists():
        raise FileNotFoundError(f"Missing file: {fpath}")
    size_mb = fpath.stat().st_size / 1_048_576
    print(f"  ✓ {fpath.name:<35} {size_mb:6.1f} MB")

print(f"\n✓ All splits downloaded successfully.")
print(f"Splits dir   : {SPLITS_DIR}")
print(f"Output will be saved to S3: {S3_OUTPUT_BUCKET}")

## Step 4 — Load Split & Inspect Class Distribution

Loads the train and val arrays for `FOLD_K`. Prints shape, dtype, and a class distribution summary to verify balance.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

# ── Load label map ────────────────────────────────────────────────────────
label_map_path = SPLITS_DIR / "label_map.json"
with open(label_map_path, "r", encoding="utf-8") as fh:
    label_map = {int(k): v for k, v in json.load(fh).items()}

N_CLASSES     = len(label_map)
species_names = [label_map[i] for i in range(N_CLASSES)]

print(f"Classes      : {N_CLASSES}")

# ── Load train / val arrays ───────────────────────────────────────────────
train_data = np.load(str(SPLITS_DIR / f"split_{FOLD_K}_train.npz"))
val_data   = np.load(str(SPLITS_DIR / f"split_{FOLD_K}_val.npz"))

X_train, y_train = train_data["spectrograms"], train_data["labels"]
X_val,   y_val   = val_data["spectrograms"],   val_data["labels"]

print(f"Train shape  : {X_train.shape}  dtype={X_train.dtype}")
print(f"Val shape    : {X_val.shape}    dtype={y_val.dtype}")
print(f"Val %        : {100 * len(y_val) / (len(y_train) + len(y_val)):.1f}%")

# ── Class distribution bar chart ──────────────────────────────────────────
train_counts = np.bincount(y_train, minlength=N_CLASSES)
val_counts   = np.bincount(y_val,   minlength=N_CLASSES)

fig_h = max(6, N_CLASSES * 0.28)
fig, ax = plt.subplots(figsize=(10, fig_h))
y_pos   = np.arange(N_CLASSES)
ax.barh(y_pos,        train_counts, height=0.5, label="Train", color="steelblue")
ax.barh(y_pos + 0.5,  val_counts,   height=0.5, label="Val",   color="tomato", alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([s.replace("_", " ") for s in species_names], fontsize=7)
ax.invert_yaxis()
ax.set_xlabel("Chunk count")
ax.set_title(f"Class distribution — fold {FOLD_K}")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Train range  : [{train_counts.min()}, {train_counts.max()}]  mean={train_counts.mean():.1f}")
print(f"Val range    : [{val_counts.min()},   {val_counts.max()}]    mean={val_counts.mean():.1f}")

## Step 5 — Build `tf.data` Pipelines

Constructs optimised `tf.data` input pipelines:
- **Train**: shuffle → expand dims → augment (flip, brightness, contrast) → batch → prefetch
  (No caching: augmentation creates different data each epoch, so caching is inefficient)
- **Val**: expand dims → cache → batch → prefetch
  (Cached: validation data is static and small)

`expand_dims` converts each `(128, 188)` tensor to `(128, 188, 1)` before batching.
Augmentation is applied only to training data.
GPU memory growth is enabled to prevent OOM errors on limited GPU memory.

In [ ]:
import tensorflow as tf

# ── Configure GPU memory growth to prevent OOM errors ─────────────────────
# Allow TensorFlow to allocate memory as needed instead of grabbing all at once
# This is critical on SageMaker instances with limited VRAM
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"⚠ Warning: Could not set GPU memory growth: {e}")
else:
    print("ℹ No GPU detected, running on CPU")

AUTOTUNE = tf.data.AUTOTUNE

def expand_and_cast(spec, label):
    spec = tf.expand_dims(spec, axis=-1)   # (128, 188) → (128, 188, 1)
    spec = tf.cast(spec, tf.float32)
    return spec, label

def augment(spec, label):
    spec = tf.image.random_flip_left_right(spec)
    spec = tf.image.random_brightness(spec, max_delta=0.1)
    spec = tf.image.random_contrast(spec, lower=0.8, upper=1.2)
    spec = tf.clip_by_value(spec, 0.0, 1.0)
    return spec, label

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=min(SHUFFLE_BUFFER, len(y_train)))
    .map(expand_and_cast, num_parallel_calls=2)
    .map(augment, num_parallel_calls=2)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(buffer_size=2)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .map(expand_and_cast, num_parallel_calls=2)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=2)
)

print("✓ tf.data pipelines configured (memory-optimized for SageMaker).")

## Step 6 — Build Model

Architecture:
```
Input (128, 188, 1)
  → Lambda: repeat grayscale channel × 3  →  (128, 188, 3)
  → ResNet50V2 backbone (ImageNet weights, top removed)
  → GlobalAveragePooling2D
  → BatchNormalization
  → Dense(256, relu)
  → Dropout(0.4)
  → Dense(N_CLASSES, softmax)
```

The Lambda layer converts the single-channel log-Mel spectrogram to 3 channels so that the pretrained ImageNet weights can be applied directly.

In [ ]:
import tensorflow as tf

def build_model(n_classes, input_height, input_width, dropout_rate, dense_units):
    inputs = tf.keras.Input(shape=(input_height, input_width, 1), name="spectrogram")

    # Grayscale → 3-channel (required by pretrained ResNet50V2)
    # Concatenate instead of Lambda to avoid serialisation issues on load.
    x = tf.keras.layers.Concatenate(axis=-1, name="to_rgb")([inputs, inputs, inputs])

    # Pretrained backbone (weights frozen initially)
    base_model = tf.keras.applications.ResNet50V2(
        include_top=False,
        weights="imagenet",
        input_shape=(input_height, input_width, 3),
    )
    base_model.trainable = False

    x = base_model(x, training=False)

    # Classification head
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.BatchNormalization(name="head_bn")(x)
    x = tf.keras.layers.Dense(dense_units, activation="relu", name="head_dense")(x)
    x = tf.keras.layers.Dropout(dropout_rate, name="head_dropout")(x)
    outputs = tf.keras.layers.Dense(n_classes, activation="softmax", name="predictions")(x)

    model = tf.keras.Model(inputs, outputs, name="BirdSongClassifier")
    return model, base_model


model, base_model = build_model(
    n_classes=N_CLASSES,
    input_height=INPUT_HEIGHT,
    input_width=INPUT_WIDTH,
    dropout_rate=DROPOUT_RATE,
    dense_units=DENSE_UNITS,
)

model.summary(line_length=90)
print(f"\nBase model trainable : {base_model.trainable}")
print(f"Total parameters     : {model.count_params():,}")

## Step 7 — Phase 1: Train Classification Head

The ResNet50V2 base is **frozen**. Only the new classification head layers are trained.

This lets the head adapt to the spectrogram feature space before any fine-tuning disturbs the pretrained weights.

In [ ]:
import tensorflow as tf

# ── Compile ───────────────────────────────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_PHASE1),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_accuracy"),
    ],
)

# ── Callbacks ─────────────────────────────────────────────────────────────
callbacks_phase1 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINT_PATH),
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=EARLY_STOP_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE,
        min_lr=1e-7,
        verbose=1,
    ),
]

print("Phase 1 — training classification head ...")
history_phase1 = model.fit(
    train_ds,
    epochs=EPOCHS_PHASE1,
    validation_data=val_ds,
    callbacks=callbacks_phase1,
    verbose=1,
)

val_acc_p1 = max(history_phase1.history["val_accuracy"])
print(f"\nPhase 1 complete. Best val accuracy: {val_acc_p1:.4f}")

## Step 8 — Phase 2: Fine-Tune Top Layers

Unfreeze the **last `UNFREEZE_FROM` layers** of the ResNet50V2 base and continue training with a learning rate 10× lower than Phase 1 to preserve the pretrained representations.

In [ ]:
import tensorflow as tf

# ── Unfreeze top layers of base ───────────────────────────────────────────
base_model.trainable = True
for layer in base_model.layers[:UNFREEZE_FROM]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Base layers now trainable : {trainable_count} / {len(base_model.layers)}")

# ── Recompile at lower LR ─────────────────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_PHASE2),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_accuracy"),
    ],
)

callbacks_phase2 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINT_PATH),
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=EARLY_STOP_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE,
        min_lr=1e-8,
        verbose=1,
    ),
]

print("Phase 2 — fine-tuning top layers ...")
history_phase2 = model.fit(
    train_ds,
    initial_epoch=len(history_phase1.history["loss"]),
    epochs=len(history_phase1.history["loss"]) + EPOCHS_PHASE2,
    validation_data=val_ds,
    callbacks=callbacks_phase2,
    verbose=1,
)

val_acc_p2 = max(history_phase2.history["val_accuracy"])
print(f"\nPhase 2 complete. Best val accuracy: {val_acc_p2:.4f}")

## Step 9 — Evaluation & Diagnostics

Four diagnostic outputs:
1. **Learning curves** — train/val accuracy and loss across both phases
2. **Confusion matrix** — predicted vs. true species (normalised by row)
3. **Per-class accuracy** — horizontal bar chart, sorted ascending
4. **Summary metrics** — top-1 accuracy, top-5 accuracy, macro F1, weighted F1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, f1_score, top_k_accuracy_score
)

# ── 1. Learning curves (phases 1 + 2 combined) ───────────────────────────
acc      = history_phase1.history["accuracy"]     + history_phase2.history["accuracy"]
val_acc  = history_phase1.history["val_accuracy"] + history_phase2.history["val_accuracy"]
loss     = history_phase1.history["loss"]         + history_phase2.history["loss"]
val_loss = history_phase1.history["val_loss"]     + history_phase2.history["val_loss"]
phase_boundary = len(history_phase1.history["loss"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Training curves — fold {FOLD_K}", fontsize=13)

for ax, train_vals, val_vals, ylabel in zip(
    axes,
    [acc, loss],
    [val_acc, val_loss],
    ["Accuracy", "Loss"],
):
    epochs = range(1, len(train_vals) + 1)
    ax.plot(epochs, train_vals, label="Train")
    ax.plot(epochs, val_vals,   label="Val")
    ax.axvline(phase_boundary, color="gray", linestyle="--", linewidth=1.2, label="Phase 2 start")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(str(MODELS_DIR / f"fold{FOLD_K}_learning_curves.png"), dpi=120)
plt.show()

# ── 2. Predictions on val set ─────────────────────────────────────────────
print("Running inference on validation set ...")
y_prob = model.predict(val_ds, verbose=0)   # (M, N_CLASSES)
y_pred = np.argmax(y_prob, axis=1)

# ── 3. Confusion matrix ───────────────────────────────────────────────────
cm     = confusion_matrix(y_val, y_pred, normalize="true")
labels = [s.replace("_", " ") for s in species_names]
fig_sz = max(12, N_CLASSES * 0.4)

fig, ax = plt.subplots(figsize=(fig_sz, fig_sz))
sns.heatmap(cm, annot=N_CLASSES <= 30, fmt=".2f",
            xticklabels=labels, yticklabels=labels,
            cmap="Blues", linewidths=0.3, ax=ax)
ax.set_xlabel("Predicted", fontsize=11)
ax.set_ylabel("True", fontsize=11)
ax.set_title(f"Confusion matrix (row-normalised) — fold {FOLD_K}", fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(str(MODELS_DIR / f"fold{FOLD_K}_confusion_matrix.png"), dpi=120)
plt.show()

# ── 4. Per-class accuracy bar chart ──────────────────────────────────────
per_class_acc = cm.diagonal()
sorted_idx    = np.argsort(per_class_acc)

fig, ax = plt.subplots(figsize=(10, max(6, N_CLASSES * 0.28)))
colors  = ["tomato" if v < 0.5 else "steelblue" for v in per_class_acc[sorted_idx]]
ax.barh(range(N_CLASSES), per_class_acc[sorted_idx], color=colors, height=0.7)
ax.set_yticks(range(N_CLASSES))
ax.set_yticklabels([labels[i] for i in sorted_idx], fontsize=7)
ax.axvline(0.5, color="gray", linestyle="--", linewidth=1.0)
ax.set_xlabel("Per-class accuracy")
ax.set_title(f"Per-class accuracy — fold {FOLD_K}")
plt.tight_layout()
plt.savefig(str(MODELS_DIR / f"fold{FOLD_K}_per_class_accuracy.png"), dpi=120)
plt.show()

# ── 5. Summary metrics ────────────────────────────────────────────────────
top1_acc    = np.mean(y_pred == y_val)
top5_acc    = top_k_accuracy_score(y_val, y_prob, k=5)
macro_f1    = f1_score(y_val, y_pred, average="macro",    zero_division=0)
weighted_f1 = f1_score(y_val, y_pred, average="weighted", zero_division=0)

print("\n" + "=" * 40)
print(f"  Top-1 accuracy : {top1_acc:.4f}")
print(f"  Top-5 accuracy : {top5_acc:.4f}")
print(f"  Macro F1       : {macro_f1:.4f}")
print(f"  Weighted F1    : {weighted_f1:.4f}")
print("=" * 40)

worst5 = [labels[i] for i in sorted_idx[:5]]
best5  = [labels[i] for i in sorted_idx[-5:]]
print(f"\nWorst 5 species: {worst5}")
print(f"Best  5 species: {best5}")

# ── 6. AUC-ROC and Precision-Recall curves ───────────────────────────────
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
)

y_val_bin = label_binarize(y_val, classes=list(range(N_CLASSES)))

fpr_all, tpr_all, roc_auc_all = {}, {}, {}
prec_all, rec_all, ap_all     = {}, {}, {}

for i in range(N_CLASSES):
    fpr_all[i], tpr_all[i], _ = roc_curve(y_val_bin[:, i], y_prob[:, i])
    roc_auc_all[i]             = auc(fpr_all[i], tpr_all[i])
    prec_all[i], rec_all[i], _ = precision_recall_curve(y_val_bin[:, i], y_prob[:, i])
    ap_all[i]                  = average_precision_score(y_val_bin[:, i], y_prob[:, i])

# Macro-average ROC
all_fpr  = np.unique(np.concatenate([fpr_all[i] for i in range(N_CLASSES)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(N_CLASSES):
    mean_tpr += np.interp(all_fpr, fpr_all[i], tpr_all[i])
mean_tpr /= N_CLASSES
macro_roc_auc = auc(all_fpr, mean_tpr)

# Macro mean-average precision
mean_ap = float(np.mean([ap_all[i] for i in range(N_CLASSES)]))

# Indices of worst / best 5 classes by AUC and AP
roc_auc_vals = np.array([roc_auc_all[i] for i in range(N_CLASSES)])
ap_vals      = np.array([ap_all[i]      for i in range(N_CLASSES)])
worst5_roc   = np.argsort(roc_auc_vals)[:5]
best5_roc    = np.argsort(roc_auc_vals)[-5:]
worst5_ap    = np.argsort(ap_vals)[:5]
best5_ap     = np.argsort(ap_vals)[-5:]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f"AUC-ROC & Precision-Recall Curves — fold {FOLD_K}", fontsize=13)

# ── ROC subplot ───────────────────────────────────────────────────────────
ax = axes[0]
for i in worst5_roc:
    ax.plot(fpr_all[i], tpr_all[i], color="tomato",    alpha=0.55, lw=1.1)
for i in best5_roc:
    ax.plot(fpr_all[i], tpr_all[i], color="steelblue", alpha=0.55, lw=1.1)
ax.plot(all_fpr, mean_tpr, color="navy", lw=2.2,
        label=f"Macro-avg (AUC = {macro_roc_auc:.3f})")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve (One-vs-Rest)")
ax.legend(fontsize=9, loc="lower right")
ax.annotate("worst 5 classes", xy=(0.55, 0.18), color="tomato",    fontsize=8)
ax.annotate("best 5 classes",  xy=(0.05, 0.88), color="steelblue", fontsize=8)

# ── PR subplot ────────────────────────────────────────────────────────────
ax = axes[1]
for i in worst5_ap:
    ax.plot(rec_all[i], prec_all[i], color="tomato",    alpha=0.55, lw=1.1)
for i in best5_ap:
    ax.plot(rec_all[i], prec_all[i], color="steelblue", alpha=0.55, lw=1.1)

# Interpolated macro PR (rec_all[i] is decreasing → reverse for np.interp)
all_rec_grid     = np.linspace(0, 1, 300)
mean_prec_interp = np.zeros(300)
for i in range(N_CLASSES):
    mean_prec_interp += np.interp(all_rec_grid,
                                  rec_all[i][::-1],
                                  prec_all[i][::-1])
mean_prec_interp /= N_CLASSES

ax.plot(all_rec_grid, mean_prec_interp, color="navy", lw=2.2,
        label=f"Macro-avg (mAP = {mean_ap:.3f})")
ax.axhline(1.0 / N_CLASSES, color="gray", linestyle="--", lw=1,
           label=f"Random baseline ({1.0 / N_CLASSES:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve (One-vs-Rest)")
ax.legend(fontsize=9, loc="upper right")
ax.annotate("worst 5 classes", xy=(0.5,  0.05), color="tomato",    fontsize=8)
ax.annotate("best 5 classes",  xy=(0.05, 0.88), color="steelblue", fontsize=8)

plt.tight_layout()
plt.savefig(str(MODELS_DIR / f"fold{FOLD_K}_auc_roc_pr.png"), dpi=120)
plt.show()

print(f"  Macro AUC-ROC      : {macro_roc_auc:.4f}")
print(f"  Mean Avg Precision : {mean_ap:.4f}")

## Step 10 — Save Model

Saves the trained model in Keras native format. The checkpoint with the best validation accuracy (saved during training) is already loaded via `restore_best_weights=True` in the EarlyStopping callback.

In [ ]:
model.save(str(FINAL_MODEL_PATH))
print(f"Model saved locally: {FINAL_MODEL_PATH}")

# ── Upload model to S3 ────────────────────────────────────────────────────
print(f"\nUploading model to S3...")
model_filename = f"resnet50v2_fold{FOLD_K}.keras"
s3_model_key   = f"{s3_output_prefix}{model_filename}".lstrip("/")

try:
    s3_client.upload_file(
        str(FINAL_MODEL_PATH),
        s3_output_bucket,
        s3_model_key
    )
    s3_model_uri = f"s3://{s3_output_bucket}/{s3_model_key}"
    print(f"✓ Model uploaded: {s3_model_uri}")
except Exception as e:
    print(f"✗ S3 upload failed: {e}")
    raise

# ── Checkpoint also uploaded ──────────────────────────────────────────────
checkpoint_filename = f"fold{FOLD_K}_best.weights.h5"
s3_checkpoint_key   = f"{s3_output_prefix}{checkpoint_filename}".lstrip("/")
if CHECKPOINT_PATH.exists():
    try:
        s3_client.upload_file(
            str(CHECKPOINT_PATH),
            s3_output_bucket,
            s3_checkpoint_key
        )
        s3_checkpoint_uri = f"s3://{s3_output_bucket}/{s3_checkpoint_key}"
        print(f"✓ Checkpoint uploaded: {s3_checkpoint_uri}")
    except Exception as e:
        print(f"⚠ Checkpoint upload failed (non-critical): {e}")

print(f"\n✓ Training complete!")
print(f"  Model files:")
print(f"    Local  : {FINAL_MODEL_PATH}")
print(f"    S3     : {s3_model_uri}")

# ── To restore the model ──────────────────────────────────────────────────
print(f"\nTo restore the model from S3:")
print(f"  import boto3; import tensorflow as tf")
print(f"  s3 = boto3.client('s3')")
print(f"  s3.download_file('{s3_output_bucket}', '{s3_model_key}', 'model.keras')")
print(f"  model = tf.keras.models.load_model('model.keras')")

## Step 11 — Inference Helper (SageMaker / S3 Compatible)

Given any raw audio file, runs the full pre-processing chain and returns the **top-3 predicted species** with confidence scores.

Can load the trained model from local disk OR download from S3 via boto3.
The pre-processing parameters (SR, chunk size, bandpass filter, mel settings, normalisation) match those used in `pre-processing.ipynb` Steps 6 and 8.

> **Configuration:** Set `USE_S3_MODEL = True` to load the model from S3. Otherwise, points to a local `MODEL_LOCAL_PATH`.

In [ ]:
%pip install librosa soundfile scipy boto3 pandas pyarrow --quiet

import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import boto3
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────
# Model can be loaded from local path OR downloaded from S3.
DEFAULT_RUN_NAME = globals().get("RUN_NAME", "index_split")
MODEL_LOCAL_PATH = Path(globals().get("FINAL_MODEL_PATH", f"resnet50v2_{DEFAULT_RUN_NAME}.keras"))

# S3: Bucket and key for trained model.
S3_MODEL_BUCKET = globals().get("s3_output_bucket", "my-bucket")  # ← UPDATE if running standalone
S3_MODEL_KEY = f"{globals().get('s3_output_prefix', '')}{MODEL_LOCAL_PATH.name}".lstrip("/")
USE_S3_MODEL = False                                             # ← Set to True to download from S3

# ── Audio file for inference ──────────────────────────────────────────────
# Can be a local file path OR an S3 URI (e.g. "s3://my-bucket/path/to/bird_audio.mpeg")
AUDIO_PATH = "s3://my-bucket/path/to/bird_audio.mpeg"

# ── Metadata artifacts: dataset index and normalization stats ─────────────
def default_artifact_key(relative_key):
    if "artifact_key" in globals():
        return artifact_key(relative_key)
    return str(relative_key).lstrip("/")

DATASET_INDEX_LOCAL_PATH = Path(globals().get("DATASET_INDEX_PATH", "data/dataset_index_balanced.parquet"))
NORM_STATS_LOCAL_PATH    = Path(globals().get("NORM_STATS_PATH", "data/norm_stats.parquet"))
S3_METADATA_BUCKET       = globals().get("s3_feature_bucket", "my-bucket")       # ← UPDATE if running standalone
S3_DATASET_INDEX_KEY     = default_artifact_key(globals().get("DATASET_INDEX_KEY", "data/dataset_index_balanced.parquet"))
S3_NORM_STATS_KEY        = default_artifact_key(globals().get("NORM_STATS_KEY", "data/norm_stats.parquet"))
USE_S3_METADATA          = False                                                 # ← Set to True to download metadata

# ── Load metadata artifacts ────────────────────────────────────────────────
if USE_S3_METADATA:
    s3 = boto3.client("s3")
    local_index = Path("/tmp/dataset_index_balanced.parquet")
    local_stats = Path("/tmp/norm_stats.parquet")
    print(f"Downloading dataset index: s3://{S3_METADATA_BUCKET}/{S3_DATASET_INDEX_KEY}")
    s3.download_file(S3_METADATA_BUCKET, S3_DATASET_INDEX_KEY, str(local_index))
    print(f"Downloading norm stats   : s3://{S3_METADATA_BUCKET}/{S3_NORM_STATS_KEY}")
    s3.download_file(S3_METADATA_BUCKET, S3_NORM_STATS_KEY, str(local_stats))
    dataset_index_path = local_index
    norm_stats_path = local_stats
else:
    dataset_index_path = DATASET_INDEX_LOCAL_PATH
    norm_stats_path = NORM_STATS_LOCAL_PATH

if not dataset_index_path.exists():
    raise FileNotFoundError(f"Dataset index not found at {dataset_index_path}")
if not norm_stats_path.exists():
    raise FileNotFoundError(f"Normalization stats not found at {norm_stats_path}")

df_index = pd.read_parquet(dataset_index_path)
label_table = (
    df_index[["label", "scientific_name"]]
    .drop_duplicates(subset=["label"])
    .sort_values("label")
)
label_map = {
    int(row.label): row.scientific_name
    for row in label_table.itertuples(index=False)
}
print(f"✓ Labels loaded from dataset index ({len(label_map)} classes)")

norm_df = pd.read_parquet(norm_stats_path).sort_values("mel_bin")
norm_mean = norm_df["mean"].to_numpy(dtype=np.float32)
norm_std = np.maximum(norm_df["std"].to_numpy(dtype=np.float32), 1e-6)
print(f"✓ Normalization stats loaded ({len(norm_mean)} Mel bins)")

# ── Load Model ────────────────────────────────────────────────────────────
if USE_S3_MODEL:
    print(f"\nDownloading model from S3: s3://{S3_MODEL_BUCKET}/{S3_MODEL_KEY}")
    s3 = boto3.client("s3")
    local_model_path = Path("/tmp/resnet50v2_model.keras")
    s3.download_file(S3_MODEL_BUCKET, S3_MODEL_KEY, str(local_model_path))
    model_path = local_model_path
else:
    model_path = MODEL_LOCAL_PATH

if not model_path.exists():
    raise FileNotFoundError(f"Model not found at {model_path}")

print(f"Loading model from {model_path.name}...")
model = tf.keras.models.load_model(str(model_path))
print("✓ Model loaded successfully")

# ── Handle S3 Audio Retrieval if URI is provided ──────────────────────────
local_audio_path = AUDIO_PATH
source_label = Path(AUDIO_PATH).name if AUDIO_PATH else "Unknown"

if AUDIO_PATH and str(AUDIO_PATH).startswith("s3://"):
    # Extract bucket and key from the s3:// URI
    s3_uri = str(AUDIO_PATH).replace("s3://", "")
    parts = s3_uri.split("/", 1)
    if len(parts) != 2:
        raise ValueError(f"Invalid S3 URI: {AUDIO_PATH}")
    s3_audio_bucket, s3_audio_key = parts[0], parts[1]
    
    filename = Path(s3_audio_key).name
    local_audio_path = Path("/tmp") / filename
    local_audio_path.parent.mkdir(parents=True, exist_ok=True)
    
    print(f"S3 Audio URI detected. Downloading s3://{s3_audio_bucket}/{s3_audio_key} → {local_audio_path}...")
    s3 = boto3.client("s3")
    try:
        s3.download_file(s3_audio_bucket, s3_audio_key, str(local_audio_path))
        print("✓ Audio file downloaded from S3 successfully.")
        source_label = f"S3: {filename}"
    except Exception as e:
        print(f"✗ Failed to download audio from S3: {e}")
        raise
else:
    local_audio_path = Path(AUDIO_PATH) if AUDIO_PATH else None

# ── Verify audio file ─────────────────────────────────────────────────────
if local_audio_path is None or not Path(local_audio_path).exists():
    raise ValueError(f"Please set AUDIO_PATH to an existing local file or valid S3 URI. Current value: {AUDIO_PATH}")

# ── Feature constants matching pre_processing.ipynb Block 6 ───────────────
SR             = 22_050
CHUNK_LEN      = 2.6
MIN_CHUNK_LEN  = 1.0
CHUNK_SAMPLES  = int(SR * CHUNK_LEN)
STRIDE_SAMPLES = CHUNK_SAMPLES // 2
N_FFT          = 1024
HOP_LENGTH     = 256
N_MELS         = 128
F_MIN          = 300.0
F_MAX          = 10_000.0
TOP_DB         = 80
TARGET_FRAMES  = 220

if len(norm_mean) != N_MELS or len(norm_std) != N_MELS:
    raise ValueError(f"Expected {N_MELS} norm-stat rows, got mean={len(norm_mean)}, std={len(norm_std)}")

# ── Feature extraction ────────────────────────────────────────────────────
def audio_to_chunks(audio):
    if len(audio) < int(MIN_CHUNK_LEN * SR):
        return []
    if len(audio) < CHUNK_SAMPLES:
        padded = np.zeros(CHUNK_SAMPLES, dtype=np.float32)
        padded[:len(audio)] = audio
        return [padded]

    chunks = []
    for start in range(0, len(audio), STRIDE_SAMPLES):
        chunk = audio[start:start + CHUNK_SAMPLES]
        if len(chunk) == CHUNK_SAMPLES:
            chunks.append(chunk.astype(np.float32))
        elif len(chunk) >= int(MIN_CHUNK_LEN * SR):
            padded = np.zeros(CHUNK_SAMPLES, dtype=np.float32)
            padded[:len(chunk)] = chunk
            chunks.append(padded)
            break
        else:
            break
    return chunks


def chunk_to_tensor(chunk):
    mel = librosa.feature.melspectrogram(
        y=chunk,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=F_MIN,
        fmax=F_MAX,
        power=2.0,
    )
    log_mel = librosa.power_to_db(mel, ref=np.max, top_db=TOP_DB)

    if log_mel.shape[1] < TARGET_FRAMES:
        log_mel = np.pad(
            log_mel,
            ((0, 0), (0, TARGET_FRAMES - log_mel.shape[1])),
            mode="constant",
            constant_values=log_mel.min(),
        )
    else:
        log_mel = log_mel[:, :TARGET_FRAMES]

    tensor = (log_mel - norm_mean[:, None]) / norm_std[:, None]
    return tensor[:, :, np.newaxis].astype(np.float32)

# ── Inference ─────────────────────────────────────────────────────────────
audio, _ = librosa.load(str(local_audio_path), sr=SR, mono=True)
chunks = audio_to_chunks(audio)

if not chunks:
    raise ValueError("Audio is too short to extract any valid chunks.")

tensors_in = np.stack([chunk_to_tensor(c) for c in chunks], axis=0)

print(f"\nInference on {source_label}")
print(f"  Chunks      : {len(chunks)}")
print(f"  Tensor shape: {tensors_in.shape}")

# ── Predict ───────────────────────────────────────────────────────────────
chunk_probs = model.predict(tensors_in, verbose=0)  # (K, N_CLASSES)
avg_probs = chunk_probs.mean(axis=0)                # average across chunks

top3_indices = np.argsort(avg_probs)[::-1][:3]

print("\n── Top-3 predicted species ──────────────────")
for rank, idx in enumerate(top3_indices, 1):
    species = label_map[int(idx)].replace("_", " ")
    confidence = avg_probs[idx] * 100
    print(f"  {rank}. {species:<40}  {confidence:5.1f}%")
print("─────────────────────────────────────────────")